#### Import, Load & Clean, Connect Neo4j

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.graph_builder import GraphBuilder
from src.data_loader import load_data
from src.preprocessing import clean_acquisitions, clean_acquiring_companies, clean_acquired_companies

In [2]:
acquisitions, acquiring_companies, acquired_companies = load_data()

acquisitions = clean_acquisitions(acquisitions)
acquiring_companies = clean_acquiring_companies(acquiring_companies)
acquired_companies = clean_acquired_companies(acquired_companies)

In [3]:
import os

uri = os.getenv("NEO4J_URI", "bolt://127.0.0.1:7687")
user = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD", "enterprise-kg")

builder = GraphBuilder(uri, user, password)

#### NODES Creation

In [4]:
for _, row in acquiring_companies.iterrows():
    builder.create_company(
        name=row["Acquiring Company"],
        year_founded=row["Year Founded"],
        market_categories=row["Market Categories"],
        country=row["Country (HQ)"],
        city=row["City (HQ)"],
        description=row["Description"],
        homepage=row["Homepage"],
        twitter=row["Twitter"],
        total_funding=row["Total Funding ($)"],
        number_of_employees=row["Number of Employees"]
    )

for _, row in acquired_companies.iterrows():
    builder.create_company(
        name=row["Company"],
        year_founded=row["Year Founded"],
        market_categories=row["Market Categories"],
        country=row["Country (HQ)"],
        city=row["City (HQ)"],
        description=row["Description"],
        homepage=row["Homepage"],
        twitter=row["Twitter"]
    )

In [5]:
for _, row in acquisitions.iterrows():
    builder.create_acquisition(
        acquirer=row["Acquiring Company"],
        target=row["Acquired Company"],
        acquisition_id=row["Acquisitions ID"],
        year=row["Year of acquisition announcement"],
        price=row["Price"],
        status=row["Status"],
        terms=row["Terms"]
    )

##### Verifying Acquisition Relationships

In [6]:
with builder.driver.session() as session:
    node_count = session.run("MATCH (c:Company) RETURN count(c) AS count").single()["count"]
    rel_count = session.run("MATCH ()-[r:ACQUIRED]->() RETURN count(r) AS count").single()["count"]
    print(f"Company nodes: {node_count}")
    print(f"ACQUIRED relationships: {rel_count}")

Company nodes: 1698
ACQUIRED relationships: 1690


In [7]:
from src.entity_extractor import extract_and_store

extract_and_store(builder)

Processing 50 company descriptions...
  Extracted: Microsoft → ACQUIRED → Tellme
  Extracted: Palm → ACQUIRED → HP
  Extracted: Amicima → ACQUIRED → Adobe
  Extracted: Behance → ACQUIRED → AdWeek
  Extracted: Adobe → ACQUIRED → EchoSign
  Extracted: Efficient Frontier → ACQUIRED → Context Optional
  Extracted: Emerald City Software → ACQUIRED → Adobe Systems Inc.
  Extracted: Macromedia → ACQUIRED → Adobe Systems
  Extracted: Adobe → ACQUIRED → Scene7
  Extracted: Amazon.com → ACQUIRED → Cadabra
  Extracted: Brilliance Audio, Inc. → ACQUIRED → Amazon.com, Inc.
  Extracted: Amazon → ACQUIRED → dpreview.com
  Extracted: LOVEFiLM → ACQUIRED → Amazon EU SARL

Done. 13 new relationships extracted and stored.


In [8]:
builder.close()